# 实验二 · 基线训练任务（Transformer 英汉翻译）

一键流程：环境确认 → 训练 10 epoch（batch 4096，每 5000 iter 存 checkpoint）→
预测 newstest2017/2018 → BLEU 评测 → 打包产物。

预计耗时约 2.5 小时（V100 单卡）。训练日志实时重定向到 run_logs/train_baseline.log。


In [ ]:
%%bash
set -x
date
nvidia-smi | head -12
python -c "import paddle; print('paddle', paddle.__version__, 'cuda', paddle.is_compiled_with_cuda())"
ls -la
mkdir -p run_logs output_file


In [ ]:
%%bash
set -o pipefail
date
python -u main.py \
  --do_train True \
  --epoch 10 \
  --src_vocab_fpath wmt18_zhen_data_bpe3/vocab_all.bpe.32000 \
  --trg_vocab_fpath wmt18_zhen_data_bpe3/vocab_all.bpe.32000 \
  --special_token '<s>' '<e>' '<unk>' \
  --training_file wmt18_zhen_data_bpe3/train.tok.clean.bpe.32000.en-zh \
  --batch_size 4096 \
  --save_step 5000 \
  --print_step 100 \
  2>&1 | tee run_logs/train_baseline.log
echo "TRAIN_EXIT=$?"
date


In [ ]:
%%bash
set -o pipefail
python -u main.py \
  --do_predict True \
  --src_vocab_fpath wmt18_zhen_data_bpe3/vocab_all.bpe.32000 \
  --trg_vocab_fpath wmt18_zhen_data_bpe3/vocab_all.bpe.32000 \
  --special_token '<s>' '<e>' '<unk>' \
  --predict_file wmt18_zhen_data_bpe3/newstest2017.tok.bpe.32000.en-zh \
  --batch_size 64 \
  --init_from_params saved_models/step_final/ \
  --beam_size 5 \
  --max_out_len 255 \
  --output_file output_file/predict_2017.txt \
  2>&1 | tee run_logs/predict_2017.log
echo "PREDICT_EXIT=$?"


In [ ]:
%%bash
set -o pipefail
python -u main.py \
  --do_predict True \
  --src_vocab_fpath wmt18_zhen_data_bpe3/vocab_all.bpe.32000 \
  --trg_vocab_fpath wmt18_zhen_data_bpe3/vocab_all.bpe.32000 \
  --special_token '<s>' '<e>' '<unk>' \
  --predict_file wmt18_zhen_data_bpe3/newstest2018.tok.bpe.32000.en-zh \
  --batch_size 64 \
  --init_from_params saved_models/step_final/ \
  --beam_size 5 \
  --max_out_len 255 \
  --output_file output_file/predict_2018.txt \
  2>&1 | tee run_logs/predict_2018.log
echo "PREDICT_EXIT=$?"


In [ ]:
%%bash
for f in predict_2017 predict_2018; do
  sed -r 's/(@@ )|(@@ ?$)//g' output_file/$f.txt > output_file/$f.tok.txt
done
perl mosesdecoder/scripts/generic/multi-bleu.perl wmt18_zhen_data_bpe3/newstest2017.tok.zh < output_file/predict_2017.tok.txt 2>&1 | tee run_logs/bleu_2017.log
perl mosesdecoder/scripts/generic/multi-bleu.perl wmt18_zhen_data_bpe3/newstest2018.tok.zh < output_file/predict_2018.tok.txt 2>&1 | tee run_logs/bleu_2018.log


In [ ]:
%%bash
zip -qr -1 baseline_outputs.zip run_logs output_file
zip -qr -1 saved_models.zip saved_models
ls -la baseline_outputs.zip saved_models.zip
du -sh saved_models/
ls saved_models/
